# Data Cleaning & Enrichment Pipeline
## FAOSTAT Producer Prices — Australia & New Zealand (1991–2025)

**Subject:** Data Visualisation Design and Storytelling — Group Assignment (Parts 2 & 3)  
**Team:** Analysts — Chaitya Nanavati & Shreyas Yadugani  
**Narrative arc:** *The Sparkline* — What Is (current food price volatility) vs What Could Be (stable, food-secure futures)  
**Stakeholder:** UN Food Systems Summit Panel  

---

## Dataset Overview

| # | Source | Role in Story | Join Key | Coverage |
|---|--------|---------------|----------|----------|
| 1 | **FAOSTAT Producer Prices** | Primary dataset | `iso3` + `year` | 1991–2025, AUS & NZL, 120 commodities |
| 2 | **FAO Food Price Index (FFPI)** | Global price benchmark — *the storm* | `year` | 1990–2026, monthly & annual |
| 3 | **Global Hunger Index (GHI)** | Hunger context — *who suffers* | `iso3` | 136 countries, 2000/2008/2016/2025 |
| 4 | **World Bank Food Import %** | Trade vulnerability — *who is exposed* | `iso3` + `year` | 1960–2024, 260+ countries |

---

## How Enrichment Sources Are Used

**FFPI (Enrichment 1):** The FAO Food Price Index provides a globally normalised price benchmark (base 2014–2016 = 100). Joined on `year`, it lets the dashboard show when AUS/NZ commodity prices diverge from or amplify global shocks (2008 food crisis, 2011 Arab Spring spike, 2022 Ukraine war surge). The monthly sheet powers spike annotation overlays on the time-series visual.

**GHI (Enrichment 2):** The Global Hunger Index quantifies hunger severity by country. AUS and NZL are high-income nations not ranked by GHI — their `ghi_2025` is `NaN` by design. The full GHI table is used for the world hunger context choropleth panel. The narrative link: commodities exported from AUS/NZ reach import-dependent, hunger-vulnerable nations — when prices spike, those nations feel it most.

**World Bank (Enrichment 3):** Food imports as % of merchandise imports (indicator `TM.VAL.FOOD.ZS.UN`). Joined on `iso3 + year`, this quantifies which countries are most exposed when AUS/NZ producer prices rise. Powers the what-if scenario slider: *'If wheat prices rise 20%, which nations face the highest compounded risk?'*


---
## 0. Imports & Setup


In [ ]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from pathlib import Path
import os
import warnings
warnings.filterwarnings('ignore')

# Resolve paths from the repository root so the notebook works after clone.
def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'README.md').exists() and (candidate / 'data').exists():
            return candidate
    return start.resolve()

ROOT_DIR = find_repo_root(Path.cwd())
RAW_DIR = ROOT_DIR / 'data' / 'raw'
OUT_DIR = ROOT_DIR / 'data' / 'processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Libraries loaded.')
print(f'Raw data directory: {RAW_DIR}')
print(f'Output directory: {OUT_DIR}')


---
## 1. FAOSTAT Producer Prices

**File:** `faostat_producer_prices_aus_nzl.csv`  
**Encoding:** Latin-1 (FAO standard export — not UTF-8)  
**Countries:** Australia, New Zealand  
**Elements included:** USD/tonne · LCU/tonne · SLC/tonne · Price Index (2014-2016=100)  


### 1.1 Raw load


In [ ]:
df_fao = pd.read_csv(RAW_DIR / 'faostat_producer_prices_aus_nzl.csv',encoding='latin-1',keep_default_na=False)

# Fix BOM character on column 0
df_fao.rename(columns={df_fao.columns[0]: 'Domain Code'}, inplace=True)

print('Raw shape:', df_fao.shape)
print('Columns:', df_fao.columns.tolist())
df_fao.head(3)


### 1.2 Rename columns & cast types


In [ ]:
df_fao.rename(columns={
    'Area':             'country',
    'Item':             'item',
    'Year':             'year',
    'Months':           'month',
    'Value':            'value',
    'Unit':             'unit',
    'Flag':             'flag',
    'Flag Description': 'flag_description',
    'Element':          'element',
}, inplace=True)

df_fao['year']  = pd.to_numeric(df_fao['year'],  errors='coerce').astype('Int64')
df_fao['value'] = pd.to_numeric(df_fao['value'], errors='coerce')

print('Countries:  ', df_fao['country'].unique().tolist())
print('Year range: ', df_fao['year'].min(), '–', df_fao['year'].max())
print('Elements breakdown:')
print(df_fao['element'].value_counts())
print('Flag codes found:')
print(df_fao['flag'].value_counts())


### 1.3 Add ISO3 country codes (primary join key)

FAO uses its own country name strings. We map them to ISO 3166-1 alpha-3 codes,
which are the standard join key for all enrichment sources.


In [ ]:
ISO3_MAP = {
    'Australia':   'AUS',
    'New Zealand': 'NZL',
    # Extend this dict if future FAOSTAT downloads include more countries
}

df_fao['iso3'] = df_fao['country'].map(ISO3_MAP)

unmapped = df_fao[df_fao['iso3'].isna()]['country'].unique()
if len(unmapped) == 0:
    print('All countries mapped to ISO3.')
else:
    print(f'Unmapped: {unmapped}')  # must be empty before joining


### 1.4 Split by element type


In [ ]:
# Separate the 4 element types for clarity.
# USD/tonne - cross-country comparison (primary for dashboard)
# Price Index - relative trend analysis (2014-2016 = 100)
# LCU/tonne - raw local-currency value (kept for reference)

df_usd = df_fao[df_fao['element'] == 'Producer Price (USD/tonne)'].copy()
df_idx = df_fao[df_fao['element'] == 'Producer Price Index (2014-2016 = 100)'].copy()
df_lcu = df_fao[df_fao['element'] == 'Producer Price (LCU/tonne)'].copy()

print(f'USD/tonne rows:   {len(df_usd):,}')
print(f'Price Index rows: {len(df_idx):,}')
print(f'LCU/tonne rows:   {len(df_lcu):,}')


### 1.5 Handle flags, missing values & zeros

| Flag | Meaning | Action |
|------|---------|--------|
| `A` | Official figure | ✅ Keep |
| `E` | FAO estimate | ✅ Keep — mark `is_imputed = True` |
| `X` | International reliable source | ✅ Keep — mark `is_imputed = True` |
| Zero | Not meaningful as a price | ❌ Drop |
| NaN | No value recorded | ❌ Drop |


In [ ]:
def clean_fao(df):
    d = df.copy()
    d['is_imputed'] = d['flag'].isin(['E', 'X'])
    d = d[(d['value'] > 0) & d['value'].notna()].copy() # Drop zero prices and NaN values
    return d

df_usd = clean_fao(df_usd)
df_idx = clean_fao(df_idx)
df_lcu = clean_fao(df_lcu)

print('After cleaning:')
for name, d in [('USD', df_usd), ('IDX', df_idx), ('LCU', df_lcu)]:
    print(f'  {name}: {d.shape}  |  imputed rows: {d["is_imputed"].sum()}')


### 1.6 Outlier detection — IQR method per (country, commodity) pair

> **Important:** We FLAG outliers but do NOT drop them.  
> The 2008 food crisis, 2011 Arab Spring spike, and 2022 Ukraine war surge are **real events**
> and are central to the Sparkline narrative. Removing them would destroy the story.


In [ ]:
def add_outlier_flag(df):
    """Flag values beyond ±3×IQR within each (iso3, item) group."""
    d = df.copy()
    d['is_outlier'] = d.groupby(['iso3', 'item'])['value'].transform(
        lambda x: (
            (x < x.quantile(0.25) - 3 * (x.quantile(0.75) - x.quantile(0.25))) |
            (x > x.quantile(0.75) + 3 * (x.quantile(0.75) - x.quantile(0.25)))
        )
    )
    return d

df_usd = add_outlier_flag(df_usd)
df_idx = add_outlier_flag(df_idx)
df_lcu = add_outlier_flag(df_lcu)

print('Outliers flagged (not dropped):')
for name, d in [('USD', df_usd), ('IDX', df_idx), ('LCU', df_lcu)]:
    print(f'  {name}: {d["is_outlier"].sum()} rows flagged')
print()
print('Sample outlier rows (USD):')
df_usd[df_usd['is_outlier']][['country','item','year','value']].head(8)


---
## 2. FAO Food Price Index (FFPI) — Enrichment Source 1

**File:** `FAO_Food_Price_Index_.xlsx`  
**Base:** 2014–2016 = 100  
**Sheets used:** `Indices_Monthly_Nominal` (monthly) · `Annual`  

**Narrative use:** Overlaying the FFPI on AUS/NZ producer prices reveals whether local prices
track, lag, or amplify global shocks. The 2022 FFPI reached its highest level since records began.


In [ ]:
wb_ffpi = load_workbook(RAW_DIR / 'fao_food_price_index.xlsx', read_only=True)
print('Sheets:', wb_ffpi.sheetnames)


### 2.1 Monthly nominal indices (1990–2026)


In [ ]:
FFPI_COLS = ['ffpi_food', 'ffpi_meat', 'ffpi_dairy', 'ffpi_cereals', 'ffpi_oils', 'ffpi_sugar']

rows_m = []
for i, row in enumerate(wb_ffpi['Indices_Monthly_Nominal'].iter_rows(values_only=True)):
    if i < 3 or row[0] is None:
        continue
    rows_m.append({
        'date': row[0],
        **{FFPI_COLS[j]: row[j + 1] for j in range(6)}
    })

df_ffpi_m = pd.DataFrame(rows_m)
df_ffpi_m['date']      = pd.to_datetime(df_ffpi_m['date'])
df_ffpi_m['year']      = df_ffpi_m['date'].dt.year.astype('Int64')
df_ffpi_m['month_num'] = df_ffpi_m['date'].dt.month
for c in FFPI_COLS:
    df_ffpi_m[c] = pd.to_numeric(df_ffpi_m[c], errors='coerce')

print(f'Shape: {df_ffpi_m.shape}')
print(f'Date range: {df_ffpi_m["date"].min().date()} → {df_ffpi_m["date"].max().date()}')
df_ffpi_m.head(4)


### 2.2 Annual indices — join key for FAOSTAT enrichment


In [ ]:
rows_a = []
for i, row in enumerate(wb_ffpi['Annual'].iter_rows(values_only=True)):
    if i < 3 or row[0] is None:
        continue
    rows_a.append({
        'year': row[0],
        **{FFPI_COLS[j]: row[j + 1] for j in range(6)}
    })

df_ffpi_annual = pd.DataFrame(rows_a)
df_ffpi_annual['year'] = pd.to_numeric(df_ffpi_annual['year'], errors='coerce').astype('Int64')
for c in FFPI_COLS:
    df_ffpi_annual[c] = pd.to_numeric(df_ffpi_annual[c], errors='coerce')

print(f'Shape: {df_ffpi_annual.shape}')
print(f'Year range: {df_ffpi_annual["year"].min()} – {df_ffpi_annual["year"].max()}')
print()
# Show key crisis years
crisis_years = [2008, 2011, 2022]
print('Key crisis years (FFPI Food Index):')
df_ffpi_annual[df_ffpi_annual['year'].isin(crisis_years)][['year','ffpi_food']]


---
## 3. Global Hunger Index (GHI) — Enrichment Source 2

**File:** `global_hunger_index.xlsx` — sheet: *GHI Scores 2025*  
**Coverage:** 136 countries, scores for 2000 · 2008 · 2016 · 2025  
**Scale:** 0 = no hunger, 100 = extremely alarming hunger  

**Narrative use:** Provides the human cost layer. Countries with high GHI scores are often heavily
import-dependent — when AUS/NZ commodity prices rise, these are the nations that suffer most.
AUS and NZL are **not in GHI rankings** (high-income nations) — `ghi_2025 = NaN` is correct.


In [ ]:
wb_ghi = load_workbook(RAW_DIR / 'global_hunger_index.xlsx', read_only=True)
print('Sheets:', wb_ghi.sheetnames)


In [ ]:
# Parse GHI Scores sheet
ghi_rows = []
for i, row in enumerate(wb_ghi['GHI Scores 2025 '].iter_rows(values_only=True)):
    if i < 3 or row[0] is None:
        continue
    ghi_rows.append({
        'country_ghi':    row[0],
        'ghi_2000':       row[1],
        'ghi_2008':       row[2],
        'ghi_2016':       row[3],
        'ghi_2025':       row[4],
        'ghi_abs_change': row[5],
        'ghi_pct_change': row[6],
    })

df_ghi = pd.DataFrame(ghi_rows)

# Drop footnote rows
df_ghi = df_ghi[df_ghi['country_ghi'].astype(str).str.len() < 60]

# GHI uses '<5' for very low scores — coerce to NaN (low hunger, not a problem for our story)
def safe_float(x):
    try: return float(x)
    except: return np.nan

for c in ['ghi_2000', 'ghi_2008', 'ghi_2016', 'ghi_2025']:
    df_ghi[c] = df_ghi[c].apply(safe_float)

print(f'GHI records: {df_ghi.shape}')
df_ghi.head(5)


### 3.1 Map GHI country names → ISO3


In [ ]:
GHI_ISO3 = {
    'Afghanistan':'AFG', 'Albania':'ALB', 'Algeria':'DZA', 'Angola':'AGO',
    'Argentina':'ARG', 'Armenia':'ARM', 'Azerbaijan':'AZE', 'Bangladesh':'BGD',
    'Belarus':'BLR', 'Bolivia (Plurinat. State of)':'BOL',
    'Bosnia & Herzegovina':'BIH', 'Botswana':'BWA', 'Brazil':'BRA',
    'Bulgaria':'BGR', 'Burkina Faso':'BFA', 'Burundi':'BDI',
    'Cabo Verde':'CPV', 'Cambodia':'KHM', 'Cameroon':'CMR',
    'Central African Republic':'CAF', 'Chad':'TCD', 'Chile':'CHL',
    'China':'CHN', 'Colombia':'COL', 'Comoros':'COM',
    'Congo (Republic of)':'COG', 'Costa Rica':'CRI', 'Croatia':'HRV',
    "Côte d'Ivoire":'CIV', 'Dem. Rep. of the Congo':'COD',
    'Djibouti':'DJI', 'Dominican Republic':'DOM', 'Ecuador':'ECU',
    'Egypt':'EGY', 'El Salvador':'SLV', 'Equatorial Guinea':'GNQ',
    'Eritrea':'ERI', 'Estonia':'EST', 'Eswatini':'SWZ', 'Ethiopia':'ETH',
    'Fiji':'FJI', 'Gabon':'GAB', 'Gambia':'GMB', 'Georgia':'GEO',
    'Ghana':'GHA', 'Guatemala':'GTM', 'Guinea':'GIN', 'Guinea-Bissau':'GNB',
    'Guyana':'GUY', 'Haiti':'HTI', 'Honduras':'HND', 'Hungary':'HUN',
    'India':'IND', 'Indonesia':'IDN', 'Iran (Islamic Republic of)':'IRN',
    'Iraq':'IRQ', 'Jamaica':'JAM', 'Jordan':'JOR', 'Kazakhstan':'KAZ',
    'Kenya':'KEN', 'Korea (DPR)':'PRK', 'Kuwait':'KWT', 'Kyrgyzstan':'KGZ',
    'Lao PDR':'LAO', 'Latvia':'LVA', 'Lebanon':'LBN', 'Lesotho':'LSO',
    'Liberia':'LBR', 'Libya':'LBY', 'Lithuania':'LTU', 'Madagascar':'MDG',
    'Malawi':'MWI', 'Maldives':'MDV', 'Malaysia':'MYS', 'Mali':'MLI',
    'Mauritania':'MRT', 'Mauritius':'MUS', 'Mexico':'MEX',
    'Moldova (Rep. of)':'MDA', 'Mongolia':'MNG', 'Montenegro':'MNE',
    'Morocco':'MAR', 'Mozambique':'MOZ', 'Myanmar':'MMR', 'Namibia':'NAM',
    'Nepal':'NPL', 'Nicaragua':'NIC', 'Niger':'NER', 'Nigeria':'NGA',
    'North Macedonia':'MKD', 'Oman':'OMN', 'Pakistan':'PAK', 'Panama':'PAN',
    'Papua New Guinea':'PNG', 'Paraguay':'PRY', 'Peru':'PER',
    'Philippines':'PHL', 'Qatar':'QAT', 'Romania':'ROU',
    'Russian Federation':'RUS', 'Rwanda':'RWA', 'Saudi Arabia':'SAU',
    'Senegal':'SEN', 'Serbia':'SRB', 'Sierra Leone':'SLE', 'Slovakia':'SVK',
    'Solomon Islands':'SLB', 'Somalia':'SOM', 'South Africa':'ZAF',
    'South Sudan':'SSD', 'Sri Lanka':'LKA', 'Sudan':'SDN', 'Suriname':'SUR',
    'Syrian Arab Republic':'SYR', 'Tajikistan':'TJK',
    'Tanzania (United Rep. of)':'TZA', 'Thailand':'THA', 'Timor-Leste':'TLS',
    'Togo':'TGO', 'Trinidad & Tobago':'TTO', 'Tunisia':'TUN',
    'Turkmenistan':'TKM', 'Türkiye':'TUR', 'Uganda':'UGA', 'Ukraine':'UKR',
    'United Arab Emirates':'ARE', 'Uruguay':'URY', 'Uzbekistan':'UZB',
    'Venezuela (Boliv. Rep. of)':'VEN', 'Viet Nam':'VNM',
    'Yemen':'YEM', 'Zambia':'ZMB', 'Zimbabwe':'ZWE',
    'Bahrain':'BHR', 'Benin':'BEN', 'Bhutan':'BTN',
    'Australia':'AUS', 'New Zealand':'NZL',
}

df_ghi['iso3'] = df_ghi['country_ghi'].map(GHI_ISO3)

n_mapped   = df_ghi['iso3'].notna().sum()
n_unmapped = df_ghi['iso3'].isna().sum()
print(f'Mapped:   {n_mapped}/{len(df_ghi)}')
print(f'Unmapped: {n_unmapped} - {df_ghi[df_ghi["iso3"].isna()]["country_ghi"].tolist()}')


### 3.2 Top 10 most food-insecure countries (GHI 2025)
These are the nations most likely to be harmed when global food prices spike.


In [ ]:
top_hungry = (df_ghi[df_ghi['ghi_2025'].notna()]
    .nlargest(10, 'ghi_2025')[['country_ghi','iso3','ghi_2025','ghi_2016','ghi_2000']]
    .reset_index(drop=True)
)
top_hungry


---
## 4. World Bank Food Import Dependency — Enrichment Source 3

**File:** `worldbank_food_import_raw.csv`  
**Indicator:** `TM.VAL.FOOD.ZS.UN` — Food imports as % of merchandise imports  
**Format:** Wide (countries × year columns) → melted to long

**Narrative use:** High food import dependency + high commodity prices = vulnerability.
This powers the what-if scenario: *'If AUS/NZ wheat prices rise X%, these countries' food
import bills increase by Y% — and they already spend Z% of export earnings on food.'*


In [ ]:
df_wb_raw = pd.read_csv(RAW_DIR / 'worldbank_food_import_raw.csv', skiprows=4, encoding='utf-8-sig')
print('Shape (wide):', df_wb_raw.shape)
print('Columns sample:', df_wb_raw.columns[:6].tolist())


In [ ]:
# Identify year columns
year_cols = [c for c in df_wb_raw.columns if str(c).strip().isdigit()]
print(f'Year columns found: {len(year_cols)} ({year_cols[0]} – {year_cols[-1]})')

df_wb = df_wb_raw[['Country Name', 'Country Code'] + year_cols].melt(
    id_vars=['Country Name', 'Country Code'],
    var_name='year',
    value_name='food_import_pct'
)
df_wb.rename(columns={'Country Name': 'country_wb','Country Code': 'iso3',}, inplace=True)

df_wb['year']            = pd.to_numeric(df_wb['year'], errors='coerce').astype('Int64')
df_wb['food_import_pct'] = pd.to_numeric(df_wb['food_import_pct'], errors='coerce')
df_wb = df_wb.dropna(subset=['food_import_pct'])

print(f'Long format shape: {df_wb.shape}')
print(f'Countries: {df_wb["iso3"].nunique()}')
print(f'Year range: {df_wb["year"].min()} – {df_wb["year"].max()}')
df_wb.head(4)


In [ ]:
# AUS and NZL food import dependency over time
print('Australia food import % (last 10 years):')
print(df_wb[df_wb['iso3']=='AUS'].sort_values('year').tail(10)[['year','food_import_pct']].to_string(index=False))
print()
print('New Zealand food import % (last 10 years):')
print(df_wb[df_wb['iso3']=='NZL'].sort_values('year').tail(10)[['year','food_import_pct']].to_string(index=False))


### 4.1 Top 15 most food-import-dependent countries (latest year available)


In [ ]:
latest_wb = ( df_wb.sort_values('year', ascending=False)
    .groupby('iso3')
    .first()
    .reset_index()
    .nlargest(15, 'food_import_pct')
    [['iso3', 'country_wb', 'year', 'food_import_pct']]
    .reset_index(drop=True)
)
latest_wb


---
## 5. Enrich: Join All Sources onto FAOSTAT

Three left joins — left joins preserve ALL FAOSTAT rows even if enrichment data is missing:

```
FAOSTAT (USD/tonne)                         - base table
   LEFT JOIN  FFPI Annual    ON year        - global price benchmark
   LEFT JOIN  World Bank     ON iso3+year   - food import dependency
   LEFT JOIN  GHI 2025       ON iso3        - hunger severity snapshot
```


In [ ]:
# Prepare slim versions for joining
ghi_slim = df_ghi[['iso3', 'ghi_2025']].dropna(subset=['iso3'])
wb_slim  = df_wb[['iso3', 'year', 'food_import_pct']]

def enrich(df):
    """Apply all three enrichment joins to a FAOSTAT dataframe."""
    return (
        df
        .merge(df_ffpi_annual, on='year',           how='left')  # Enrichment 1: FFPI
        .merge(wb_slim,        on=['iso3', 'year'], how='left')  # Enrichment 3: World Bank
        .merge(ghi_slim,       on='iso3',           how='left')  # Enrichment 2: GHI
    )

master_usd = enrich(df_usd)
master_idx = enrich(df_idx)

print(f'Master USD enriched: {master_usd.shape}')
print(f'Master IDX enriched: {master_idx.shape}')
print()
print('Null check on joined columns:')
null_report = master_usd[['food_import_pct', 'ghi_2025', 'ffpi_food']].isnull().sum()
print(null_report)
print()
print('Note: ghi_2025 is NaN for AUS/NZL — expected (high-income nations not in GHI ranking)')


In [ ]:
# Preview the fully enriched master dataset
master_usd[['country', 'iso3', 'item', 'year', 'value','ffpi_food', 'food_import_pct', 'ghi_2025','is_imputed', 'is_outlier'
]].head(10)


---
## 6. Data Quality Checks


In [ ]:
print('MASTER PRODUCER PRICES (USD/tonne)')
print(f'  Rows:          {len(master_usd):,}')
print(f'  Countries:     {master_usd["country"].unique().tolist()}')
print(f'  Commodities:   {master_usd["item"].nunique()} unique items')
print(f'  Year range:    {master_usd["year"].min()} – {master_usd["year"].max()}')
print(f'  Outlier rows:  {master_usd["is_outlier"].sum()}')
print(f'  Imputed rows:  {master_usd["is_imputed"].sum()}')
print()
print('Value summary (USD/tonne):')
print(master_usd['value'].describe().round(2))


In [ ]:
# FFPI coverage — are all FAOSTAT years covered?
fao_years  = set(master_usd['year'].dropna().astype(int))
ffpi_years = set(df_ffpi_annual['year'].dropna().astype(int))
missing = sorted(fao_years - ffpi_years)
print(f'FAOSTAT years not in FFPI: {missing if missing else "None — full coverage"}')


In [ ]:
# Top 10 most expensive commodities on average
print('Top 10 highest average producer prices (USD/tonne):')
(
    master_usd.groupby('item')['value']
    .mean()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
    .rename(columns={'value': 'avg_usd_per_tonne'})
    .assign(avg_usd_per_tonne=lambda x: x['avg_usd_per_tonne'].round(2))
)


In [ ]:
# Price trend check — AUS wheat (a key global export)
wheat = master_usd[
    (master_usd['country']=='Australia') &
    (master_usd['item'].str.contains('Wheat', case=False))
].sort_values('year')[['year','value','ffpi_food','food_import_pct']]

if len(wheat) > 0:
    print('Australia Wheat prices vs FFPI (sample years):')
    print(wheat.iloc[::5].to_string(index=False))  # every 5th row
else:
    print('Note: Wheat not in this FAOSTAT slice — check item list.')
    print('Available items sample:', master_usd['item'].unique()[:10].tolist())


In [ ]:
# Outlier flagged for dashboard
outliers = master_usd[master_usd['is_outlier']][['country','item','year','value']]
print(f'Flagged outliers ({len(outliers)} rows):')
outliers.sort_values('value', ascending=False)


---
## 7. Save All Output Files

| File | Description | Primary use in dashboard |
|------|-------------|-------------------------|
| `master_producer_prices_usd.csv` | Primary enriched dataset | All 4 visuals |
| `master_producer_price_index.csv` | Price index (2014-2016=100) + enrichments | Trend / sparkline visual |
| `master_producer_prices_lcu.csv` | LCU/tonne (local currency, cleaned) | Reference only |
| `ffpi_monthly.csv` | FFPI monthly 1990–2026 | Spike annotation timeline |
| `ffpi_annual.csv` | FFPI annual 1990–2025 | Year-on-year comparison |
| `ghi_cleaned.csv` | GHI scores 136 countries | Hunger context choropleth |
| `worldbank_food_import_pct.csv` | Food import % all countries | What-if scenario slider |


In [ ]:
file_map = {
    'master_producer_prices_usd.csv':  master_usd,
    'master_producer_price_index.csv': master_idx,
    'master_producer_prices_lcu.csv':  df_lcu,
    'ffpi_monthly.csv':                df_ffpi_m,
    'ffpi_annual.csv':                 df_ffpi_annual,
    'ghi_cleaned.csv':                 df_ghi,
    'worldbank_food_import_pct.csv':   df_wb,
}

for fname, df in file_map.items():
    path = OUT_DIR / fname
    df.to_csv(path, index=False)

---
## 8. Data Dictionary

### `master_producer_prices_usd.csv` — primary enriched file (27 columns)

| Column | Type | Source | Description |
|--------|------|--------|-------------|
| `country` | string | FAOSTAT | FAO country name |
| `iso3` | string | Derived | ISO 3166-1 alpha-3 code (join key) |
| `item` | string | FAOSTAT | Commodity name (e.g. 'Wheat', 'Apples') |
| `year` | Int64 | FAOSTAT | Observation year (1991–2025) |
| `month` | string | FAOSTAT | 'Annual value' for yearly obs; month name for monthly |
| `value` | float | FAOSTAT | Producer price in USD per tonne (cleaned: >0, non-null) |
| `unit` | string | FAOSTAT | 'USD' |
| `flag` | string | FAOSTAT | Data quality: A=official, E=estimated, X=intl sources |
| `is_imputed` | bool | Derived | True if flag ∈ {E, X} |
| `is_outlier` | bool | Derived | True if value > Q3+3×IQR or < Q1−3×IQR within (iso3, item) |
| `ffpi_food` | float | FAO FFPI | FAO Food Price Index (2014-2016=100), annual, global |
| `ffpi_meat` | float | FAO FFPI | Meat Price Index |
| `ffpi_dairy` | float | FAO FFPI | Dairy Price Index |
| `ffpi_cereals` | float | FAO FFPI | Cereals Price Index |
| `ffpi_oils` | float | FAO FFPI | Vegetable Oils Price Index |
| `ffpi_sugar` | float | FAO FFPI | Sugar Price Index |
| `food_import_pct` | float | World Bank | Food imports as % of merchandise imports |
| `ghi_2025` | float | GHI | Hunger Index score 2025 (0–100); NaN for AUS/NZL |

---

### Join logic
```
FAOSTAT USD  LEFT JOIN  FFPI Annual    ON  year
             LEFT JOIN  World Bank     ON  iso3 + year
             LEFT JOIN  GHI 2025       ON  iso3
```
---
*Pipeline authored by: Chaitya Nanavati & Shreyas Yadugani (Analysts)*  